In [3]:
import pandas as pd
from math import exp
import numpy as np
df = pd.read_csv('first_25000_rows.csv')
df

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,2024-10-21T11:54:29.221230963Z,2024-10-21T11:54:29.221064336Z,10,2,38,C,B,1,233.62,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
1,2024-10-21T11:54:29.223936626Z,2024-10-21T11:54:29.223769812Z,10,2,38,A,B,0,233.67,2,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
2,2024-10-21T11:54:29.225196809Z,2024-10-21T11:54:29.225030400Z,10,2,38,A,B,0,233.67,3,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
3,2024-10-21T11:54:29.712600612Z,2024-10-21T11:54:29.712434212Z,10,2,38,A,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
4,2024-10-21T11:54:29.764839221Z,2024-10-21T11:54:29.764673165Z,10,2,38,C,B,2,233.52,200,...,155,1,7,233.25,234.13,55,400,2,1,AAPL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,2024-10-21T13:04:16.583694069Z,2024-10-21T13:04:16.583527688Z,10,2,38,A,B,2,233.46,200,...,105,1,2,233.25,234.50,55,63,2,4,AAPL
4996,2024-10-21T13:04:17.976627074Z,2024-10-21T13:04:17.976461017Z,10,2,38,A,A,1,233.69,200,...,105,1,2,233.25,234.50,55,63,2,4,AAPL
4997,2024-10-21T13:04:20.085804687Z,2024-10-21T13:04:20.085638629Z,10,2,38,C,B,2,233.46,200,...,105,2,2,233.24,234.50,1,63,1,4,AAPL
4998,2024-10-21T13:04:20.085817362Z,2024-10-21T13:04:20.085651109Z,10,2,38,A,B,3,233.44,200,...,105,1,2,233.25,234.50,55,63,2,4,AAPL


In [4]:
df['ts_recv'] = pd.to_datetime(df['ts_recv'])
df = df.sort_values('ts_recv').reset_index(drop=True)

best_bid = df['bid_px_00'].values
best_ask = df['ask_px_00'].values
best_bid_size = df['bid_sz_00'].values
best_ask_size = df['ask_sz_00'].values

ofi = np.zeros(len(df))

for i in range(1, len(df)):

    if best_bid[i] > best_bid[i-1]:
        bid_comp = best_bid_size[i]
    elif best_bid[i] < best_bid[i-1]:
        bid_comp = -best_bid_size[i-1]
    else:
        bid_comp = best_bid_size[i] - best_bid_size[i-1]

    if best_ask[i] < best_ask[i-1]:
        ask_comp = -best_ask_size[i]
    elif best_ask[i] > best_ask[i-1]:
        ask_comp = best_ask_size[i-1]
    else:
        ask_comp = -(best_ask_size[i] - best_ask_size[i-1])

    ofi[i] = bid_comp + ask_comp


print(ofi)

[  0.   2.   3. ...   0.   0. -14.]


In [5]:
def calculate_multi_level_ofi(df, levels=10, decay_factor=0.5):

    ofi = np.zeros(len(df))

    weights = [exp(-decay_factor * level) for level in range(levels)]

    for level in range(levels):
        bid_px = df[f'bid_px_{level:02d}'].values
        ask_px = df[f'ask_px_{level:02d}'].values
        bid_sz = df[f'bid_sz_{level:02d}'].values
        ask_sz = df[f'ask_sz_{level:02d}'].values

        for i in range(1, len(df)):
            if bid_px[i] > bid_px[i-1]:
                bid_comp = bid_sz[i] * weights[level]
            elif bid_px[i] < bid_px[i-1]:
                bid_comp = -bid_sz[i-1] * weights[level]
            else:
                bid_comp = (bid_sz[i] - bid_sz[i-1]) * weights[level]

            if ask_px[i] < ask_px[i-1]:
                ask_comp = -ask_sz[i] * weights[level]
            elif ask_px[i] > ask_px[i-1]:
                ask_comp = ask_sz[i-1] * weights[level]
            else:
                ask_comp = -(ask_sz[i] - ask_sz[i-1]) * weights[level]

            ofi[i] += bid_comp + ask_comp

    return ofi

multi_level_ofi = calculate_multi_level_ofi(df, levels=10, decay_factor=0.5)
print(multi_level_ofi)

[   0.            2.            3.         ... -100.49992545   68.20311684
 -261.38123727]


In [6]:
def calculate_integrated_ofi(df, window=10):
    ofi = np.zeros(len(df))
    bpx, apx = df['bid_px_00'].values, df['ask_px_00'].values
    bsz, asz = df['bid_sz_00'].values, df['ask_sz_00'].values

    for i in range(1, len(df)):
        if bpx[i] > bpx[i-1]:
            bid_comp = bsz[i]
        elif bpx[i] < bpx[i-1]:
            bid_comp = -bsz[i-1]
        else:
            bid_comp = bsz[i] - bsz[i-1]

        if apx[i] < apx[i-1]:
            ask_comp = -asz[i]
        elif apx[i] > apx[i-1]:
            ask_comp = asz[i-1]
        else:
            ask_comp = -(asz[i] - asz[i-1])

        ofi[i] = bid_comp + ask_comp

    integrated_ofi = np.convolve(ofi, np.ones(window), 'full')[:len(ofi)]
    return integrated_ofi


integrated_ofi = calculate_integrated_ofi(df, window=10)
print(integrated_ofi)

[ 0.  2.  5. ... 29. 31. 17.]


In [7]:
df['ts_event'] = pd.to_datetime(df['ts_event'])

df = df.sort_values('ts_event')

df['delta_bid'] = df['bid_sz_00'].diff()
df['delta_ask'] = df['ask_sz_00'].diff()

df['ofi_bid'] = np.where(df['delta_bid'] > 0, df['delta_bid'], 0)
df['ofi_ask'] = np.where(df['delta_ask'] > 0, -df['delta_ask'], 0)
df['ofi'] = df['ofi_bid'] + df['ofi_ask']

ofi_result = df[['ts_event', 'bid_sz_00', 'ask_sz_00', 'delta_bid', 'delta_ask', 'ofi']].dropna()

ofi_result.to_csv('ofi_results.csv', index=False)

print(ofi_result.head(10))

                              ts_event  bid_sz_00  ask_sz_00  delta_bid  \
1  2024-10-21 11:54:29.223769812+00:00        141        200        2.0   
2  2024-10-21 11:54:29.225030400+00:00        144        200        3.0   
3  2024-10-21 11:54:29.712434212+00:00        144        200        0.0   
4  2024-10-21 11:54:29.764673165+00:00        144        200        0.0   
5  2024-10-21 11:54:29.764685027+00:00        144        200        0.0   
6  2024-10-21 11:54:36.289427977+00:00        144        200        0.0   
7  2024-10-21 11:54:37.990793976+00:00        144        200        0.0   
8  2024-10-21 11:54:39.124291588+00:00        144          1        0.0   
9  2024-10-21 11:54:39.134140341+00:00        144        200        0.0   
10 2024-10-21 11:54:41.233181043+00:00        141        200       -3.0   

    delta_ask    ofi  
1         0.0    2.0  
2         0.0    3.0  
3         0.0    0.0  
4         0.0    0.0  
5         0.0    0.0  
6         0.0    0.0  
7         0.0